# 🐯 TIGeR — Corrected Cross-Domain Run (ABO)

Companion to `tiger_corrected_run.ipynb`. **This is the notebook that needed
correcting most**, because the two defects that hit hardest were both
ABO-specific.

### Why the previous ABO results cannot stand

**A6 — image repair was switched off for the entire run.**
`arbiter.t2v_policy.allowed_categories` listed the four fashion categories only.
ABO products map to `electronics / furniture / kitchen / home_decor`, so every
E2 and E3 row was force-escalated *before* schema validation or the gamma gate
was consulted. RQ3's image-repair evidence does not exist yet. The high
escalation rate was blamed on the `color` requirement (H10) and on Arbiter
underconfidence (H11); this gate sat upstream of both and is unmentioned in
every document.

**A1 — the gamma ablation never ablated anything.**
It wrote `cfg["fusion"]["gamma"]`; the router reads `cfg["arbiter"]["gamma"]`.
"Full System" and "No Gamma Gate" were the same run, which is why they both
reported 35 repaired / 444 escalated. §7.5 and H11 explain that identity as a
"perfect early exit". There was no early exit to explain.

### What was removed from the old notebook

The old cells **6b/6c** are gone rather than ported:

* Cell 6c's markdown asserted the Arbiter was **overconfident**, *"clustered
  above gamma=0.60, so the gate never fires"* — the direct opposite of H11's
  own conclusion (**underconfident**, 75.2% *below* 0.60). Both statements
  shipped, contradicting each other.
* Its success criterion — *"the new table should show Full System > No Gamma
  Gate"* — was a check A1 made impossible to pass.
* It patched the config with `yaml.dump`, which round-trips the file and
  deletes every comment in it.
* The 25th-percentile rule it applied was justified by the early-exit story
  that A1 invalidated.

The diagnostic is kept below because the *measurement* is still worth having.
The conclusion is not pre-written this time.

**Runtime:** ~30–45 min on a T4.

## Step 0 · Clone the fixed branch and install

In [ ]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone -q -b docs/fixes-backlog-audit https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!git log --oneline -1

# H7: never --force-reinstall. It rebuilds the dependency tree, pulls an
# incompatible torchvision, and every SigLIP/SDXL cell then dies with
# "operator torchvision::nms does not exist".
!pip uninstall -y tiger -q 2>/dev/null
!pip install --no-cache-dir -e ".[dev,vlm,gen]" -q
print("\n✅ installed")

## Step 1 · Guard: confirm the fixes are present

Fails in seconds if the wrong branch was cloned, rather than after 40 minutes of GPU time.

In [ ]:
import inspect
from tiger import cli
from tiger.schema import Schema, load_schema
from tiger.fusion import PROBE_TARGETS
from tiger.eval import repair_ablation as RA
from pathlib import Path

checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

src = inspect.getsource(RA.run_repair_ablations)
check("A1  gamma written to cfg['arbiter']",
      'cfg_no_gamma.setdefault("arbiter", {})["gamma"] = 0.0' in src)
check("A4  configs deep-copied", "copy.deepcopy(cfg)" in src)
check("A3  baseline emits no E4 key",
      '"E4"' not in inspect.getsource(RA.DummyArbiter.predict_proba))
check("A3b baseline is seeded",
      "seed" in inspect.signature(RA.DummyArbiter.__init__).parameters)
check("A5  every audited field scored", hasattr(RA, "truth_from_audit"))
check("A5  image repairs scored", hasattr(RA, "image_provenance"))
check("A7  --fusion flag exposed", '"--fusion"' in inspect.getsource(cli.main))
check("A8  probes scored on-target", isinstance(PROBE_TARGETS, dict) and "color" in PROBE_TARGETS)
check("D5  surface_forms available", hasattr(Schema, "surface_forms"))

# --- Phase 1: the ABO vertical migration ---
from tiger.data.import_abo import CATEGORY_MAP, VERTICALS
from tiger.data import abo_vocab

check("P1  two ABO verticals defined",
      set(VERTICALS) == {"furnishing", "accessories"} and len(CATEGORY_MAP) == 15,
      f"{len(CATEGORY_MAP)} product types")
check("P1  phone cases excluded", "CELLULAR_PHONE_CASE" not in CATEGORY_MAP)
check("P1  colour normaliser present", hasattr(abo_vocab, "normalize_color"))
check("P1  legacy config.yaml gone", not Path("configs/config.yaml").exists())

schema = load_schema("configs/schema.yaml")
dom = schema.domain("color")
check("D5  no duplicate colours in the domain",
      len(dom) == len({schema.normalize("color", v) for v in dom}),
      f"{len(dom)} entries: {dom}")

from tiger.data.synthgen import COLOR_RGB
missing = [c for c in dom if c != "multicolour" and c not in COLOR_RGB]
check("D5  synthgen can render every colour", not missing, f"missing: {missing}")

cfg = cli.load_cfg()
allowed = cfg["arbiter"]["t2v_policy"]["allowed_categories"]
missing_t2v = sorted(set(CATEGORY_MAP.values()) - set(allowed))
check("A6  every ABO category is T2V-eligible", not missing_t2v,
      f"missing: {missing_t2v}" if missing_t2v else f"{len(allowed)} allowed")
check("P1  no ABO category requires colour",
      not (set(CATEGORY_MAP.values())
           & set(schema.attributes["color"].get("required_for_categories", []))),
      "measured coverage is 45-93%; requiring it repeats H10")

width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:<{width}}  {detail}")

failed = [n for n, ok, _ in checks if not ok]
assert not failed, f"\n\nWRONG BRANCH OR STALE INSTALL. Missing: {failed}"
print("\n✅ all fixes present — safe to proceed")

## Step 2 · Test suite (162)

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## Step 3 · Pin γ and record it

γ = 0.60 is the value the *original* ABO run used, confirmed by the committed
histogram's own caption. Starting here makes the corrected numbers directly
comparable to the ones in the paper. The diagnostic later measures whether it is
the right choice for this domain — it no longer assumes the answer.

In [ ]:
import json, re, yaml
from pathlib import Path

GAMMA = 0.60          # operating point for THIS run
NOISE_SEED = 7
BASELINE_SEED = 42    # seeds the random-routing baseline (A3b)

cfg_path = Path("configs/tiger.yaml")
text = cfg_path.read_text()

# Surgical line edit, NOT yaml.safe_dump: dumping would round-trip the file and
# silently delete every comment in it, including the rationale for the T2V
# allowlist and the noise rates. Config comments are the only place several of
# these decisions are recorded.
# Replace the whole line including its trailing comment. Leaving the old
# comment behind ("lowered from 0.60 ...") next to a value of 0.60 is precisely
# the config/doc drift that produced finding C1 in the first place.
text, n = re.subn(
    r"^(\s*)gamma:.*$",
    rf"\g<1>gamma: {GAMMA}  # Eq. 22 confidence gate; pinned by tiger_corrected_run.ipynb",
    text, count=1, flags=re.M)
assert n == 1, "could not find `gamma:` in configs/tiger.yaml"

if not re.search(r"^eval:", text, flags=re.M):
    text += ("\neval:\n"
             "  # Seeds the random-routing baseline so the 'No Arbiter' row is\n"
             "  # reproducible (finding A3b).\n"
             f"  random_baseline_seed: {BASELINE_SEED}\n")
else:
    text, n = re.subn(r"^(\s*random_baseline_seed:\s*)\d+", rf"\g<1>{BASELINE_SEED}",
                      text, count=1, flags=re.M)
    assert n == 1
cfg_path.write_text(text)

cfg = yaml.safe_load(cfg_path.read_text())
assert cfg["arbiter"]["gamma"] == GAMMA
assert cfg["eval"]["random_baseline_seed"] == BASELINE_SEED

manifest = {
    "branch": "docs/fixes-backlog-audit",
    "gamma": GAMMA,
    "noise_seed": NOISE_SEED,
    "random_baseline_seed": BASELINE_SEED,
    "dismiss_threshold": cfg["arbiter"]["dismiss_threshold"],
    "t2v_allowed_categories": cfg["arbiter"]["t2v_policy"]["allowed_categories"],
    "precision_floor": cfg["sieve"]["precision_floor"],
    "clip_model": cfg["models"]["clip_model_name"],
    "independent_verifier": cfg["models"]["independent_verifier"],
    "fusion_applied": False,
}
Path("data/outputs").mkdir(parents=True, exist_ok=True)
Path("data/outputs/run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

---
## Step 4 · Download ABO from the official source

Pulled from **`s3://amazon-berkeley-objects/`**, the AWS Registry of Open Data
release — not a third-party Kaggle mirror. This is what makes the dataset
citable as *Collins et al., "ABO: Dataset and Benchmarks for Real-World 3D
Object Understanding", CVPR 2022*.

Two archives, ~3.3 GB total, no credentials required:

| Archive | Size | Contents |
|---|---|---|
| `abo-listings.tar` | 87 MB | `listings/metadata/listings_*.json.gz` |
| `abo-images-small.tar` | 3.25 GB | `images/metadata/images.csv.gz`, `images/small/**` |

**Licence:** the S3 bucket ships `LICENSE-CC-BY-4.0.txt`, while the AWS registry
entry states CC BY-NC 4.0. Read the licence file in the download and cite what
it actually says — the two sources disagree and the NC clause would matter for
any commercial claim.

The official archives are **gzipped**. The adapter previously globbed
`listings_*.json` and called plain `open()`, so the canonical source raised
"No listings_*.json files found" and only the decompressed Kaggle mirror
worked. Fixed on this branch and covered by tests.

In [ ]:
import os, subprocess, tarfile
from pathlib import Path

ABO = Path("/kaggle/working/abo")
ABO.mkdir(parents=True, exist_ok=True)
BASE = "https://amazon-berkeley-objects.s3.amazonaws.com/archives"

for name in ("abo-listings.tar", "abo-images-small.tar"):
    tar = ABO / name
    if tar.exists():
        print(f"{name}: already downloaded ({tar.stat().st_size/1e9:.2f} GB)")
        continue
    print(f"downloading {name} ...")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(tar), f"{BASE}/{name}"], check=True)
    print(f"  {tar.stat().st_size/1e9:.2f} GB")

for name in ("abo-listings.tar", "abo-images-small.tar"):
    marker = ABO / f".extracted_{name}"
    if marker.exists():
        print(f"{name}: already extracted"); continue
    print(f"extracting {name} ...")
    with tarfile.open(ABO / name) as t:
        t.extractall(ABO)
    marker.touch()

# Read the licence as shipped rather than assuming which of the two it is.
lic = next(iter(ABO.rglob("LICENSE*")), None)
print("\nlicence file:", lic.name if lic else "not found")
if lic:
    print(lic.read_text(errors="replace")[:300].strip().replace("\n", " ")[:280])

In [ ]:
from pathlib import Path

LISTINGS_DIR = next(p for p in ABO.rglob("listings/metadata") if p.is_dir())
IMAGES_CSV   = next(iter(list(ABO.rglob("images.csv.gz")) + list(ABO.rglob("images.csv"))))
IMAGES_DIR   = next(p for p in ABO.rglob("images/small") if p.is_dir())

print("listings   ", LISTINGS_DIR, f"({len(list(LISTINGS_DIR.glob('listings_*')))} files)")
print("images.csv ", IMAGES_CSV)
print("images dir ", IMAGES_DIR)

# The canonical archive is gzipped; the Kaggle mirror was not. Both are fine now.
gz = list(LISTINGS_DIR.glob("listings_*.json.gz"))
print(f"\nlayout: {'official (.json.gz)' if gz else 'decompressed (.json)'}")

In [ ]:
!python -m tiger.cli import-abo \
    --listings-dir {LISTINGS_DIR} \
    --images-csv {IMAGES_CSV} \
    --images-dir {IMAGES_DIR} \
    --max-items 20000 \
    --max-per-category 1200

### Validate the import

H3 produced 0–16 products out of 147,000 while exiting cleanly. Check the shape, not just the exit code.

In [ ]:
import json
import pandas as pd
from tiger.data.import_abo import CATEGORY_MAP, VERTICALS
from tiger.schema import load_schema

schema = load_schema("configs/schema.yaml")
df = pd.read_parquet("data/sample/products.parquet")
assert len(df) > 5000, f"only {len(df)} products imported — see H3/H4/H5"
df["a"] = df["attributes"].map(json.loads)

cov = df.groupby("category").agg(
    n=("a", "size"),
    colour=("a", lambda s: sum("color" in x for x in s) / len(s)),
    material=("a", lambda s: sum("material" in x for x in s) / len(s)),
)
cov["vertical"] = [next(v for v, m in VERTICALS.items() if c in m) for c in cov.index]
print(f"{len(df)} products\n")
print(cov.sort_values(["vertical", "n"], ascending=[True, False])
         .to_string(float_format=lambda v: f"{v:.1%}"))
print("\nsplit:", df["split"].value_counts().to_dict())

# Every row must validate, or it is flagged dirty before any repair is attempted.
inv = sum(1 for r in df.itertuples() if schema.validate_attrs(r.category, r.a))
assert inv == 0, f"{inv} schema-invalid rows — check required_for_categories (H10)"
print("\n✅ 0 schema-invalid rows")

# Local reference run: 11,839 products, 73.3% colour, 53.9% material.
print(f"colour {sum('color' in x for x in df.a)/len(df):.1%} | "
      f"material {sum('material' in x for x in df.a)/len(df):.1%}")

---
## Step 5 · Calibrate on the ABO distribution

Thresholds, router and fusion are all domain-specific and must be refitted.
`train-arbiter` runs eight noise seeds end to end — this is the slow cell.

In [ ]:
!python -m tiger.cli calibrate

In [ ]:
!python -m tiger.cli noise --seed 7

In [ ]:
!python -m tiger.cli train-arbiter

In [ ]:
!python -m tiger.cli calibrate-fusion

---
## Step 6 · The corrected repair ablation

First time this runs with image repair actually enabled for ABO categories.

In [ ]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

## Step 7 · Results

Two checks matter more than the raw numbers.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 50)

df = pd.read_csv("data/outputs/repair_ablations_summary.csv")
cols = [c for c in ["Configuration", "Repaired", "Escalated", "Total Attempted",
                    "Attr Accuracy", "Attr Cases", "T2V Accuracy", "T2V Cases",
                    "Color Accuracy", "V2T Cases"] if c in df.columns]
display(df[cols])

full = df[df["Configuration"] == "Full System"]
nog  = df[df["Configuration"].str.contains("Gamma", na=False)]

print("\n--- A1: is the gamma gate genuinely ablated now? ---")
if len(full) and len(nog):
    if (full["Repaired"].iat[0] == nog["Repaired"].iat[0]
            and full["Escalated"].iat[0] == nog["Escalated"].iat[0]):
        print("STILL IDENTICAL. Previously this was the A1 bug. If it persists on the "
              "fixed branch it is a real finding — but prove it by intersecting the "
              "escalated row_id sets (E3). Equal totals are not equal sets.")
    else:
        print("Different ✅ — the gate is being ablated for the first time.")

print("\n--- A6: did any image repair actually happen? ---")
t2v = full["T2V Cases"].iat[0] if ("T2V Cases" in full and len(full)) else 0
print(f"T2V repairs attempted in Full System: {t2v}")
print("0 means image repair is still blocked somewhere — investigate before "
      "reporting anything about RQ3." if not t2v else
      "Non-zero ✅ — cross-domain image repair ran for the first time.")

## Step 8 · Arbiter confidence diagnostic

Measures the confidence distribution. **It does not tell you what to conclude.**

The previous version asserted the answer in its own markdown before running —
and asserted it in the direction opposite to the roadmap's finding. Read the
numbers this time.

Interpreting it: mass far below γ means the router is underconfident here and
escalates most rows on principle. Mass far above means γ never binds. Neither
is automatically wrong; both are facts about a router trained on fashion noise
and applied to furniture.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tiger.arbiter import ArbiterModel, featurize, CLASSES

model = ArbiterModel.from_json(Path("data/thresholds/tiger_arbiter_model.json").read_text())
ev_files = sorted(Path("data/outputs").glob("evidence_cal_seed*.jsonl"))
assert ev_files, "run train-arbiter first"
records = [json.loads(l) for l in ev_files[-1].read_text().splitlines() if l.strip()]

max_probs, preds = [], []
for ev in records:
    p = model.predict_proba(featurize(ev))
    top = max(p, key=p.get)
    max_probs.append(p[top]); preds.append(top)
max_probs = np.array(max_probs)

GAMMA = json.loads(Path("data/outputs/run_manifest.json").read_text())["gamma"]
print(f"n={len(max_probs)}  mean={max_probs.mean():.3f}  median={np.median(max_probs):.3f}")
print(f"below gamma={GAMMA}: {(max_probs < GAMMA).mean()*100:.1f}%")
print(f"p25={np.percentile(max_probs,25):.3f}  p75={np.percentile(max_probs,75):.3f}")
print(f"predicted classes: { {c: preds.count(c) for c in CLASSES} }")

plt.figure(figsize=(11, 4))
plt.hist(max_probs, bins=30)
plt.axvline(GAMMA, color="red", ls="--", label=f"gamma = {GAMMA}")
plt.xlabel("max arbiter confidence"); plt.ylabel("count")
plt.title("ABO Arbiter Confidence Distribution (corrected run)")
plt.legend(); plt.tight_layout()
Path("paper_figures").mkdir(exist_ok=True)
plt.savefig("paper_figures/abo_confidence_corrected.png", dpi=150)
plt.show()

### If you decide to recalibrate γ

Deliberately **not** automated. The old notebook patched the config and re-ran
in one cell, which is how γ ended up with four different values across the repo
and no record of which produced which number (C1).

To try another value: change `GAMMA` in Step 3, re-run that cell so the manifest
records it, then re-run Step 6. The manifest keeps the run identifiable.

Whatever you pick, the 25th-percentile rule needs a *new* justification. Its old
one was the early-exit argument, and A1 removed that.

## Step 9 · Export

In [ ]:
!cp -r data/outputs data/thresholds data/processed paper_figures /kaggle/working/ 2>/dev/null
!cd /kaggle/working && zip -rq tiger_abo_corrected.zip outputs thresholds processed paper_figures
!ls -la /kaggle/working/tiger_abo_corrected.zip
print("\n✅ Download tiger_abo_corrected.zip from the right sidebar.")

---
## After this run

1. **§7.5 of `paper_draft_materials.md` and H11 of `ROADMAP_PROGRESS.md` must be
   rewritten or withdrawn** (E3). They analyse an identity that A1 manufactured.
2. **§7.1's table is void** — those numbers came from a run with image repair
   disabled. Replace them with this run's.
3. **RQ3 can be answered properly for the first time.** Whatever the T2V column
   says here is the first real cross-domain image-repair evidence.
4. **H10's story survives but needs re-checking.** The `color` schema fix was a
   genuine finding; it just was not the only cause of the escalation rate.